# CFB League Commissioner Notebook

Run a hosted SEC / Big Ten / ACC fantasy league end-to-end using the FFPy database (no web UI required).

**Prerequisites:**
```bash
make cfb-full-data SEASON=2024
make db.cfb-projections-v2 SEASON=2024
```

Weekly refresh before scoring:
```bash
make db.cfb-stats SEASON=2024
make db.cfb-fantasy SEASON=2024
make db.cfb-projections-v2 SEASON=2024 WEEK=6
```

**Tip:** If you hit lineup optimizer errors, restart the kernel and run all cells — the draft fills required positions (QB/RB/WR/TE/K/DST) before best-available picks.

In [1]:
import json
import sys
import uuid
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd if (_cwd / "src" / "ffpy").exists() else _cwd.parent.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from ffpy.cfb_projections import CfbProjectionModel
from ffpy.database import FFPyDatabase
from ffpy.optimizer import LineupOptimizer, Player, RosterConstraints

SEASON = 2024
PROJECTION_MODEL = "opponent_adj"
CONFERENCES = ["SEC", "Big Ten", "ACC"]
NUM_TEAMS = 4
ROSTER_SIZE = 15

db = FFPyDatabase()
print(f"Database: {db.db_path}")

Database: /home/ricka/.ffpy/ffpy.db


## 1. Create league and teams

In [2]:
roster_path = REPO_ROOT / "config" / "roster" / "college_standard.json"
scoring_path = REPO_ROOT / "config" / "scoring" / "college_standard.json"
roster_json = roster_path.read_text()
scoring_json = scoring_path.read_text()

league_id = f"cfb:{uuid.uuid4().hex[:12]}"
db.create_cfb_league(
    {
        "league_id": league_id,
        "user_id": "notebook-commissioner",
        "name": "Notebook SEC/B1G/ACC League",
        "season": SEASON,
        "allowed_conferences": json.dumps(CONFERENCES),
        "scoring_json": scoring_json,
        "roster_slots_json": roster_json,
        "num_teams": NUM_TEAMS,
        "playoff_weeks": json.dumps([15, 16]),
        "fcs_discount_pct": 0.75,
    }
)

team_ids = []
for i in range(NUM_TEAMS):
    tid = f"{league_id}:team:{uuid.uuid4().hex[:6]}"
    db.create_cfb_league_team(
        {
            "league_team_id": tid,
            "league_id": league_id,
            "team_name": f"Team {i + 1}",
            "owner_name": f"Owner {i + 1}",
        }
    )
    team_ids.append(tid)

print(f"League {league_id} with {len(team_ids)} teams")

League cfb:ecb7260f1cd7 with 4 teams


## 2. Snake draft from player pool

In [3]:
pool = db.get_cfb_players(season=SEASON, conferences=CONFERENCES, fantasy_eligible=True)
proj = db.get_cfb_projections(SEASON, week=1, model=PROJECTION_MODEL, conferences=CONFERENCES)
if proj.empty:
    with CfbProjectionModel(db) as model:
        model.generate_projections(SEASON, 1, conferences=CONFERENCES, model=PROJECTION_MODEL)
    proj = db.get_cfb_projections(SEASON, 1, model=PROJECTION_MODEL, conferences=CONFERENCES)

draft_board = pool.merge(proj[["player_id", "projected_points"]], on="player_id", how="left")
draft_board["projected_points"] = pd.to_numeric(draft_board["projected_points"], errors="coerce").fillna(0.0)

# Fill starter minimums first so the optimizer can always field a legal lineup
STARTER_MINS = {"QB": 1, "RB": 2, "WR": 2, "TE": 1, "K": 1, "DST": 1}
FILL_ORDER = ["DST", "K", "RB", "WR", "TE", "QB"]
team_needs = {tid: STARTER_MINS.copy() for tid in team_ids}
drafted: set[int] = set()


def _best_available(position: str | None = None) -> pd.Series | None:
    avail = draft_board[~draft_board["player_id"].isin(drafted)]
    if position:
        avail = avail[avail["position"] == position]
    if avail.empty:
        return None
    return avail.sort_values("projected_points", ascending=False).iloc[0]


pick = 0
for round_num in range(ROSTER_SIZE):
    order = team_ids if round_num % 2 == 0 else list(reversed(team_ids))
    for tid in order:
        row = None
        for pos in FILL_ORDER:
            if team_needs[tid].get(pos, 0) > 0:
                row = _best_available(pos)
                if row is not None:
                    team_needs[tid][pos] -= 1
                    break
        if row is None:
            row = _best_available()
        if row is None:
            continue
        pid = int(row["player_id"])
        db.add_cfb_roster_player(tid, pid)
        drafted.add(pid)
        pick += 1

print(f"Draft complete: {pick} picks")
for tid in team_ids:
    pos_counts = db.get_cfb_league_roster(tid)["position"].value_counts().to_dict()
    print(f"  {tid}: {pos_counts}")

Draft complete: 60 picks
  cfb:ecb7260f1cd7:team:062083: {'DST': 7, 'WR': 3, 'RB': 2, 'TE': 1, 'K': 1, 'QB': 1}
  cfb:ecb7260f1cd7:team:2df1af: {'DST': 8, 'WR': 2, 'RB': 2, 'TE': 1, 'K': 1, 'QB': 1}
  cfb:ecb7260f1cd7:team:f44ed4: {'DST': 6, 'WR': 4, 'RB': 2, 'QB': 1, 'K': 1, 'TE': 1}
  cfb:ecb7260f1cd7:team:135c22: {'DST': 6, 'WR': 4, 'RB': 2, 'K': 1, 'QB': 1, 'TE': 1}


## 3. Weekly loop (lineups, matchups, scores, standings)

In [4]:
constraints = RosterConstraints.from_json_file(REPO_ROOT / "config" / "roster" / "college_standard.json")


def _build_lineup_entries(candidates: pd.DataFrame) -> list[dict]:
    """Optimize starters; keep every roster player in the projection pool."""
    id_by_key = {
        (r["full_name"], r.get("team_key") or ""): int(r["player_id"]) for _, r in candidates.iterrows()
    }
    players = [
        Player(
            name=r["full_name"],
            position=r.get("position") or "",
            team=r.get("team_key") or "",
            projected_points=float(r.get("projected_points") or 0),
        )
        for _, r in candidates.iterrows()
    ]
    if not players:
        return []

    try:
        result = LineupOptimizer(constraints=constraints).optimize(players)
        starters = result.starters
    except ValueError as exc:
        print(f"  [WARN] Optimizer fallback ({exc}) — using greedy lineup")
        starters = _greedy_starters(candidates, constraints)

    entries = []
    for p in starters:
        pid = id_by_key.get((p.name, p.team))
        if pid is None:
            continue
        entries.append({"player_id": pid, "slot": p.position or "FLEX", "is_starter": True})
    return entries


def _greedy_starters(candidates: pd.DataFrame, roster_constraints: RosterConstraints) -> list[Player]:
    """Fill required slots by best projected points when LP optimizer cannot."""
    remaining = candidates.sort_values("projected_points", ascending=False).copy()
    chosen: list[Player] = []
    used_ids: set[int] = set()

    def take(position: str | None, n: int = 1):
        nonlocal remaining
        for _ in range(n):
            pool = remaining[~remaining["player_id"].isin(used_ids)]
            if position:
                pool = pool[pool["position"] == position]
            if pool.empty:
                return
            row = pool.iloc[0]
            used_ids.add(int(row["player_id"]))
            chosen.append(
                Player(
                    name=row["full_name"],
                    position=row.get("position") or "",
                    team=row.get("team_key") or "",
                    projected_points=float(row.get("projected_points") or 0),
                )
            )

    for pos, count in roster_constraints.positions.items():
        take(pos, count)
    flex_pool = remaining[
        (~remaining["player_id"].isin(used_ids))
        & (remaining["position"].isin(roster_constraints.flex_positions))
    ]
    for _ in range(roster_constraints.num_flex):
        if flex_pool.empty:
            break
        row = flex_pool.sort_values("projected_points", ascending=False).iloc[0]
        used_ids.add(int(row["player_id"]))
        chosen.append(
            Player(
                name=row["full_name"],
                position=row.get("position") or "",
                team=row.get("team_key") or "",
                projected_points=float(row.get("projected_points") or 0),
            )
        )
        flex_pool = flex_pool[flex_pool["player_id"] != row["player_id"]]
    return chosen


def score_week(week: int):
    proj_w = db.get_cfb_projections(SEASON, week, model=PROJECTION_MODEL, conferences=CONFERENCES)
    if proj_w.empty:
        with CfbProjectionModel(db) as model:
            model.generate_projections(SEASON, week, conferences=CONFERENCES, model=PROJECTION_MODEL)
        proj_w = db.get_cfb_projections(SEASON, week, model=PROJECTION_MODEL, conferences=CONFERENCES)

    db.generate_cfb_matchups(league_id, SEASON, week)

    for tid in team_ids:
        roster = db.get_cfb_league_roster(tid)
        if roster.empty:
            continue
        # Keep full roster even if a player lacks a projection this week
        candidates = roster.merge(
            proj_w[["player_id", "projected_points"]],
            on="player_id",
            how="left",
        )
        candidates["projected_points"] = pd.to_numeric(
            candidates["projected_points"], errors="coerce"
        ).fillna(0.0)
        entries = _build_lineup_entries(candidates)
        if entries:
            db.set_cfb_lineup(tid, SEASON, week, entries)

    scored = db.score_cfb_matchups(league_id, SEASON, week)
    standings = db.get_cfb_standings(league_id, SEASON, through_week=week)
    return pd.DataFrame(scored), pd.DataFrame(standings)


w1_matchups, w1_standings = score_week(1)
w2_matchups, w2_standings = score_week(2)
display(w1_matchups[["home_team_name", "home_score", "away_team_name", "away_score"]])
display(w2_standings[["rank", "team_name", "wins", "losses", "points_for"]])

  [WARN] Optimizer fallback (No optimal solution found. Status: Infeasible) — using greedy lineup
  [WARN] Optimizer fallback (No optimal solution found. Status: Infeasible) — using greedy lineup


,home_team_name,home_score,away_team_name,away_score
0,Team 1,68.36,Team 4,12.0
1,Team 2,12.90,Team 3,15.0


,rank,team_name,wins,losses,points_for
0,1,Team 1,2,0,130.64
1,2,Team 4,1,1,37.42
2,3,Team 2,0,2,32.36
3,4,Team 3,1,1,26.80


## 4. Positional scarcity (draft analysis)

In [5]:
scarcity = (
    draft_board.groupby("position")["projected_points"]
    .agg(top12_avg=lambda s: s.head(12).mean(), depth_count="count")
    .sort_values("top12_avg", ascending=False)
)
display(scarcity)

,top12_avg,depth_count
position,,
DST,0.0,1326
K,0.0,184
QB,0.0,74
RB,0.0,193
TE,0.0,58
WR,0.0,1068


## 5. Pending transaction stub

Transactions are recorded but not processed automatically (Phase 5 stub).

In [6]:
sample_pid = int(draft_board.iloc[ROSTER_SIZE * NUM_TEAMS]["player_id"])
tx_id = db.create_cfb_transaction(
    {
        "league_id": league_id,
        "league_team_id": team_ids[0],
        "tx_type": "add",
        "player_id": sample_pid,
        "faab_bid": 11.0,
        "status": "pending",
    }
)
pd.DataFrame(db.list_cfb_transactions(league_id))

,transaction_id,league_id,league_team_id,tx_type,player_id,faab_bid,status,created_at
0,1,cfb:ecb7260f1cd7,cfb:ecb7260f1cd7:team:062083,add,2801,11.0,pending,2026-07-02 15:56:43
